# 4.11 Cluster evaluation

Cluster evaluation asks whether a discovered grouping is separated, compact, stable, and aligned with any external audit labels.

Pairwise distances produce silhouette and Davies-Bouldin scores; contingency counts produce ARI when hidden labels are available for evaluation only. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

RNG = np.random.default_rng(7)


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def scaled(X):
    return StandardScaler().fit_transform(X)


def plot_xy(X):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=0).fit_transform(X)


def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return float("nan")
    if len(np.unique(labels)) >= len(labels):
        return float("nan")
    return float(silhouette_score(X, labels))

def cluster_report(X, labels, y_true=None):
    report = {}
    report["silhouette"] = safe_silhouette(X, labels)
    if len(np.unique(labels)) > 1 and len(np.unique(labels)) < len(labels):
        report["davies_bouldin"] = float(davies_bouldin_score(X, labels))
    else:
        report["davies_bouldin"] = float("nan")
    if y_true is not None:
        report["ari"] = float(adjusted_rand_score(y_true, labels))
    else:
        report["ari"] = float("nan")
    return report


## The concept, built once on D1

The lesson evaluates a clustering with

$$s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))},\qquad DB=\frac{S_1+S_2}{M_{12}}$$

With $a=0.50$ and $b=3.00$, silhouette is $0.833$. With scatters $0.40,0.50$ and separation $3.00$, Davies-Bouldin is $0.300$.

In [ ]:

a = 0.50
b = 3.00
sil = (b - a) / max(a, b)
scatter_1 = 0.40
scatter_2 = 0.50
separation = 3.00
db = (scatter_1 + scatter_2) / separation

assert round(float(sil), 3) == 0.833
assert round(float(db), 3) == 0.300

print("silhouette audit", round(float(sil), 3))
print("Davies-Bouldin audit", round(float(db), 3))


The reusable evaluator computes silhouette, Davies-Bouldin, and optional ARI. ARI uses hidden labels only after clustering, never as training input.

In [ ]:

name_d1, X_d1, y_d1, k_d1 = cluster_ladder()[0]
Xs_d1 = scaled(X_d1)
labels_d1 = KMeans(n_clusters=k_d1, n_init=10, random_state=0).fit_predict(Xs_d1)
report_d1 = cluster_report(Xs_d1, labels_d1, y_true=y_d1)

print(name_d1)
print(report_d1)


## The dataset ladder

We use the shared F2 clustering ladder exactly once in the setup cell: hand points, clean blobs, anisotropic overlap, real Iris, and real digits 0–3. The hidden labels are kept only for audit metrics such as ARI; the clustering methods never train on them.

In [ ]:

rungs = cluster_ladder()

for idx, (name, X, y_true, k) in enumerate(rungs, start=1):
    print(idx, name, "shape=", X.shape, "k=", k, "labels=", np.unique(y_true).tolist())
    print("sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same evaluation across D1–D5

We produce a KMeans clustering for every rung, then compute silhouette, Davies-Bouldin, and ARI across the whole ladder. Silhouette is the shared metric curve; the other metrics prevent single-number tunnel vision.

In [ ]:

results = []
artifacts = []

for idx, (name, X, y_true, k) in enumerate(cluster_ladder(), start=1):
    Xs = scaled(X)
    labels = KMeans(n_clusters=k, n_init=20, random_state=idx).fit_predict(Xs)
    report = cluster_report(Xs, labels, y_true=y_true)
    row = {"rung": idx, "name": name}
    row.update(report)
    results.append(row)
    artifacts.append((Xs, labels, y_true))

for row in results:
    print(row["rung"], row["name"], "sil=", round(row["silhouette"], 3), "DB=", round(row["davies_bouldin"], 3), "ARI=", round(row["ari"], 3))


## Results visualization

Each assignment panel is annotated with silhouette, DB, and ARI; the summary curve tracks silhouette from D1 to D5.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, row, artifact in zip(axes, results, artifacts):
    Xs, labels, y_true = artifact
    coords = plot_xy(Xs)
    ax.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="tab10", s=18)
    title = f"D{row['rung']} sil={row['silhouette']:.2f}\nDB={row['davies_bouldin']:.2f} ARI={row['ari']:.2f}"
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

plt.figure(figsize=(6, 3))
plt.plot([r["rung"] for r in results], [r["silhouette"] for r in results], marker="o", label="silhouette")
plt.xticks([1, 2, 3, 4, 5])
plt.xlabel("ladder rung")
plt.ylabel("score")
plt.title("Evaluation metrics vs complexity")
plt.legend()
plt.show()


## Pitfall on D5: metric-only selection can prefer bad digit groupings

Silhouette may reward compact geometry even when clusters disagree with digit identity. The fix is a multi-metric review: silhouette, Davies-Bouldin, ARI when audit labels exist, and visual inspection.

In [ ]:

name, X5, y5, k5 = cluster_ladder()[-1]
Xs5 = scaled(X5)

rows = []

for k_try in [2, 3, 4, 5, 6, 8]:
    labels_try = KMeans(n_clusters=k_try, n_init=20, random_state=4).fit_predict(Xs5)
    report_try = cluster_report(Xs5, labels_try, y_true=y5)
    rows.append((k_try, report_try["silhouette"], report_try["davies_bouldin"], report_try["ari"]))

best_sil = max(rows, key=lambda item: item[1])
best_ari = max(rows, key=lambda item: item[3])

for row in rows:
    print("k=", row[0], "sil=", round(row[1], 3), "DB=", round(row[2], 3), "ARI=", round(row[3], 3))

print("best by silhouette", best_sil)
print("best by ARI", best_ari)


## Evaluate it + Practice

- Metric: track silhouette, Davies-Bouldin, and optional ARI on every rung and compare against a no-skill baseline such as random labels with the same number of clusters.
- Sanity check: D1 and D2 should be visibly easier than D5; if not, inspect scaling and assignments before trusting the curve.
- Ablation: select k by silhouette alone and compare with ARI or visuals
- Failure signals: metrics disagree, one cluster absorbs many digits, or a better objective creates worse external alignment
- Baseline: random labels and wrong-k KMeans

Practice prompts:
1. Change one hyperparameter by a small amount and explain whether the D5 score moves for a meaningful reason.

2. Add a random-label baseline to the results table and compare it with the method.

3. Pick one D5 point, print its assignment evidence, and decide whether the method is confident or ambiguous.